In [ ]:
import os
import sys
import torch
from pathlib import Path

if "__file__" in globals():
    project_root = Path(__file__).resolve().parent.parent
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

project_root = project_root.resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

if Path.cwd() != project_root:
    os.chdir(project_root)

from torch.utils.data import TensorDataset, random_split
from core.config import load_config, print_config
from core.data.loader import setup_dataset, load_dataset
from core.data.dataset import create_dataloaders
from core.data.transforms import create_normalizer_from_data
from core.model.bert import BertForPretraining
from core.training.sampler import create_kde_sampler
from core.training.pretrainer import setup_training
from core.logger import print_data_summary, log_model_summary

In [ ]:
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Working directory: {os.getcwd()}")

config = load_config("config", config_dir=".")
print_config(config, "Loaded BERT Configuration")

In [ ]:
colab_url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
DATASET_PATH = setup_dataset(config['pretraining']['db_path'], colab_url)

print(f"Using database: {DATASET_PATH}")

all_beatmaps_data, difficulty_attributes, loaded_ids = load_dataset(
    DATASET_PATH, 
    max_seq_len=config['data']['max_seq_len'],
    raw_beatmap_path=config['pretraining'].get('raw_beatmap_path', './data/raw')
)

print_data_summary(all_beatmaps_data)

In [ ]:
val_size = int(len(all_beatmaps_data) * config['data']['val_split'])
train_size = len(all_beatmaps_data) - val_size

temp_dataset = TensorDataset(torch.arange(len(all_beatmaps_data)))
train_split, val_split = random_split(temp_dataset, [train_size, val_size])

print(f"Data split: {len(train_split.indices)} training, {len(val_split.indices)} validation")

train_data_list = [all_beatmaps_data[i] for i in train_split.indices]
val_data_list = [all_beatmaps_data[i] for i in val_split.indices]

train_attributes = {key: val[train_split.indices] for key, val in difficulty_attributes.items()}
val_attributes = {key: val[val_split.indices] for key, val in difficulty_attributes.items()}

sampler = create_kde_sampler(
    train_attributes['stars'],
    bandwidth=config['pretraining']['sampling']['kde_bandwidth'],
    num_bins=config['pretraining']['sampling'].get('num_bins', 100),
)

normalizer = create_normalizer_from_data(train_data_list, train_attributes)
vector_stats = normalizer.get_vector_stats()

print(f"Vector normalization stats for {len(vector_stats)} fields")

In [ ]:
train_dataloader, val_dataloader = create_dataloaders(
    train_data_list,
    val_data_list,
    train_attributes,
    val_attributes,
    normalizer,
    config, device, sampler
)

print(f"Created dataloaders with batch size: {config['pretraining']['batch_size']}")

sample_batch = next(iter(train_dataloader))
print(f"Sample batch shapes: vectors={sample_batch[0].shape}, mask={sample_batch[1].shape}")
print(f"Sample attributes keys: {list(sample_batch[2].keys())}")

In [ ]:
model = BertForPretraining.from_config(config, device)
log_model_summary(model)

print("\nRunning a test forward pass with mixed precision (autocast)...")
use_amp_for_test = device.type == "cuda"
with torch.no_grad():
    with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_amp_for_test):
        sample_vectors, sample_mask, sample_attrs, sample_cu_seqlens = sample_batch
        
        sample_vectors = sample_vectors.to(device)
        sample_mask = sample_mask.to(device)

        predictions, targets, _ = model(sample_vectors, sample_mask)

        print(f"Prediction output keys: {list(predictions.keys())}")
        print(f"MLM prediction keys: {list(predictions['mlm'].keys())}")
        print(f"Difficulty prediction keys: {list(predictions['difficulty'].keys())}")

print("\nBERT model created and tested successfully!")

In [ ]:
trainer, checkpoint_manager = setup_training(
    model, train_dataloader, val_dataloader, config, device, normalizer
)

start_epoch = 0
if checkpoint_manager.checkpoint_exists():
    try:
        loaded_epoch, metrics = checkpoint_manager.load_checkpoint(
            model, trainer.optimizer, trainer.scheduler, trainer.scaler, device=device
        )
        start_epoch = loaded_epoch + 1 
        print(f"Loaded checkpoint from epoch {loaded_epoch}, resuming from epoch {start_epoch}")
        print(f"Previous metrics: {metrics}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}")
        print("Starting pretraining from scratch")

print(f"Pretraining setup complete. Starting from epoch {start_epoch}")
print(f"Total epochs: {config['pretraining']['num_epochs']}")

In [ ]:
print("\nStarting BERT pretraining...")
print(f"BERT Model: {config['model']['n_layers']} layers, {config['model']['d_model']} dimensions")

print(f"Pretraining samples: {len(train_data_list)} base maps")
print(f"Validation samples: {len(val_data_list)} base maps")

metrics_tracker = trainer.train(start_epoch)

print("\nBERT training completed!")